# Parcial práctico · Corte 2 — Los datos de un centro de distribución
**Electiva VI · Ciencia de Datos · CORHUILA 2026‑B** · Docente: Jesús Ariel González Bonilla

| | |
|---|---|
| **Modalidad** | **Individual** |
| **Valor** | 5,0 (parcial práctico del Corte 2) |
| **Tiempo estimado** | 3 a 4 horas; se puede trabajar en varias sesiones |
| **Plazo** | Desde el sábado 10 de octubre hasta el **lunes 12 de octubre de 2026, 23:59**. No se reciben entregas tardías. |
| **Entrega** | Moodle → tarea «Parcial práctico · Corte 2» → este cuaderno (.ipynb) **ejecutado, con sus salidas** (y, si se desea, un PDF del cuaderno) |
| **Herramienta** | Google Colab (recomendado) o VS Code con la extensión Jupyter |
| **Material de consulta** | Los talleres de las semanas 8 y 9 y el material del curso |

**Nombre completo del estudiante:** Camila Alejandra Salas Gracia

## Cómo está hecho este parcial
No hace falta ser programador. El cuaderno trae casi todo el código escrito y cada tarea viene **en pareja**:

| Señal | Qué es | Qué hace usted |
|---|---|---|
| 📘 **Ejemplo resuelto** | Una celda completa que resuelve un problema en **otra** tabla o columna | La ejecuta y lee qué hizo |
| 👉 **Su turno** | La misma idea, ahora sobre la tabla o columna que se indica | Cambia **poco**: un nombre, una condición o una opción. El comentario 👉 dice exactamente qué |
| 🧠 **Pregunta** | Razonar como ingeniero: qué decidir, por qué y qué significa un resultado | Responde con sus palabras y con cifras de **su** cuaderno, en la celda «Su respuesta» (doble clic para escribir) |
| ✅ / ⚠️ | Autoverificación: compara su resultado con la cifra de control **sin mostrarla** | ✅ coincide. ⚠️ todavía no: revise el cambio 👉 de esa celda |

Si una celda 👉 se deja sin cambiar, el cuaderno **sigue corriendo**: esa tarea queda en ⚠️ y no suma. **Más de la mitad de la nota está en las preguntas 🧠**: lo que más vale es explicar decisiones y leer resultados.

## Partes y puntos
| Parte | Qué se hace | Puntos |
|---|---|---|
| A · Diagnóstico | Contar cinco problemas de calidad y decidir cuál pesa más | 1,0 |
| B · Limpieza | Corregir cada problema en cinco pasos y justificar las decisiones | 1,5 |
| C · Consultas SQL | Tres consultas a partir de una plantilla y la lectura de cada resultado | 1,2 |
| D · Conclusiones | La causa del problema, una recomendación para la gerencia y un párrafo en inglés | 1,3 |
| | **Total** | **5,0** |

> ⚠️ **Antes de descargar**, use **Entorno de ejecución → Ejecutar todas** (Colab) o **Run All** (VS Code) y revise que ninguna celda termine en error. Se califica lo que se ve en el archivo entregado: código, salidas y respuestas.
>
> 💾 Colab se desconecta si se deja quieto un rato y «olvida» las variables (no las respuestas escritas). Al volver, basta con **Ejecutar todas**. Guarde una copia: **Archivo → Guardar una copia en Drive**.

## Parte 0 · Preparar el entorno
Tres celdas ya escritas. Ejecútelas en orden con **Shift + Enter** y no las modifique.
- **0.1** carga las librerías (en Colab ya vienen instaladas).
- **0.2** desempaca los datos del caso, que viajan **dentro** del cuaderno (comprimidos, como en los talleres). Funciona con o sin internet.
- **0.3** define las funciones de apoyo: `verificar` y `comprobar` (autoverificación), `registrar` (bitácora de limpieza) y `tratar_vacios` (para la Parte B).

In [1]:
# 0.1 · Librerías (se instalan solo si faltan)
import importlib.util, subprocess, sys
faltan = [lib for lib in ["pandas", "numpy"] if importlib.util.find_spec(lib) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "…")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])

import base64, gzip, json, sqlite3, hashlib, re
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
print(f"Python {sys.version.split()[0]} · pandas {pd.__version__} · listo para trabajar")

Python 3.13.2 · pandas 3.0.6 · listo para trabajar


In [2]:
#@title 0.2 · Desempacar los datos del caso (ejecútela; no la modifique)
_PAQUETE = """
H4sIAJswymoC/91dzXIcR3J+lQ6cm1Rm/Zdv0I+1suFdWuQqwjYdjhYwgsYLYhADgLGhDT2Fn2CPOujkgyN8NF/MVTPdWdlVDRbE
mZ5ZIAIhkeju6qrKvy+/zGr+5eRmvbq4P79b3b48v31/8nfNye2f7tuLxe35enlzvlxdt+fd3eJytV52bbj1/WJxsVq/vX79j398
gQDYnq7Xq5+a76+66/NVg82fLtvT77vwu7vFbXt6tXy3uA5DNxeLq+YPN8uuef3y9OXrl/S8aE/PF8u7RfN+cbm4667CCGe/aQDZ
vupu77rm9qa7/HFxd7dsNEDz2yah2tOfPvzvebfO5//l8vZuvfz+Pm7D4rb5p+7yortaXBcD6PaL7ocPvzTvVlfLi1Uxg8cOY9q3
J193t4vVbddgq5uztyft54vvlxfd7fD/5tsPv66aL6669ao5u7vo0sO2Pb287xoT3v3uqv38q8+/+fL0df0x1/7D/eWquf7wy/nd
ZgPOHv9K3365uFusL8PuLprFdXOzunrf60BcRXsa/tOcXl8sr1ebvX99v94seRgAoT3r3ndXq5+65ny9eNcNO/eoh7H94mq1Xm01
Jj7xyG1GEVTmJoz44/Jy+eGX62XQ2z+r3zaEbM8W5z8umj/+7s3m/Wfd+V0QW3v24a+bPzTfdVdXi+a7xfpikT+r2n++X9yumvPu
3c3iNi5vu+rh2UcNott/WV3eb+X1uAdP2ubkZnER9JMMfXnxH9vftD+E1XTDX+6Wi+uLrj1f3l90F2+vX3315YuwYmgFCPMC/Ivw
xzcvwgxeLYPBLu9W/S1g0y1owi1BJX+/WL7vhus+XRc4cZ29IlwK11V7za9bmcaP14UPZvfuZhUkdLleDHcZle7ScRQcvyWIn94i
40JksRCt0xAiviiIu2teXXV3j16L9+n6A2+xYrycMJvREMAmGl8hdDELm96CLg4hslmkHROChrjhQ4AevyVILp9omBjgZ+A/izfG
WzC/BUUaJUhlarlBZ9MtKt6iylGC6Dx7UbapiGwIMy2X4M9oOXHF4e/jIfR4iOw6WJPtqB8/L2xmBuV2aTbEZqG+/bpb//Thv6+H
OxRTZJhapxhPstwqwDQPUA/agnXj1QhoL8dTYYKDaC3htq9Pv/3XD//1e5KbGsstKFK+6SgzkxLFfJ0Z20t8z3gmHse6OuFg0GSL
hnwQybbFT5mlqr5Ei7HxF+J7hHhCbMzM32fbCkqPdWBizxDGulwsF0FmNlUsB5lstjZTehFvxpMNQ52dNq/OTt+c0r66bF8zsxIm
MwtRbkkSjZjyy44ZhSfjHvkqLTO/jIVo2Es2Cm/y1aKA7JbSibBbNno2sRp02SilVUg/3rMJK9Z27CrK6YJT422ZGMXIFjV3m5lP
A5PpUbFt7A6hJ63T+HFsn/B7TAfg4fBsVRbEJ3w4ZvGmNA3hsne5SSgAmSfG9ibXyrAu/fF3eZuZRzljxeSIbDrNKZ8Psli7daWF
INi6tnaUpN3Qy6AFHiXDNuaj+GyUImCDEmP3M6F5xo1FPi1PjWObzCGIsS0aPttCtWQaAdx2PVmQ0pBtb+H3nc3gmMnCOmZaN6kv
SmWboturzAUxu9+4hsyLIQtRG78woU/WjoNyxErfvDk9++bNH2gUPx4lvGq0HO1aVNzixygFhBhH9YBappZrWuR2NqUDJtMBLMK2
yCLqROwXmWef3HypMzUoB9KqEppByPHCC2QVRhhpYxn/wyMg+N5mcUq5TBnLEMNA6MadxoCaBwczDg7BuM7LPfFhIMklVGy/wxa5
MyhNgyHWwS1PvMnqzEBKmGBwnFhN+MFHCEhlAso0l+HzDZTMd9/oLLvL3I2UYy0q14ERVo1c/kQC6LPA4QoXi1nCU7g1ZsNbsDqh
BCqb7KSHZQAcGHRepT3FbE8z5+fgozkNsLRqC9+LtSjMHGOhZajGYzxg42K84BCdpsK2yGQsM7Apxns/4ZR0lvdObiwLkxvLmPA4
4MfQqYi14FyW2JTa5HTNRuu6gsyzbRddpHLKZ+68VDjns8kW0IGRDmIc3yJ/dLW8XnS3PU/EWKTN79tEJ0UG+by7vltedP3vuvT3
i8XtTXf+Y3cRBj0L2xriIUmNSN2gGRK218NKKTcnvjXoG+LmugzhhWAP0XMRjJrN9QhuKYmh8YNiod6O74brfrgedHyTnvXj65Yo
MHp/SOGV3I4f8AuhQGLnfOv7t7uW0gGiTePs1fbtQfTkgel6sB7cPh8HJeRBNHGAgGrYneG6pdljGyK76lcffAzFLdqdoOko+t0f
rofUmCjh4EWl7N8/7D4Oux8fDkNsrws9zN/R+lSr4s/2eRx2B1Ta3bDo7dNKtJTmEI8d7dL313VLNkSrDxsut7JHP4xumG7I+NPP
XhazV2z3gKQTVIIqGq51W8lr15LHoben2UWqgOyXZh/+3MtOQv92TKMbppnSYa65oJhuoKPViTR70/ZPm2F0IdNVGX/62ZuWYhVJ
No0et4TywzS74Md7zTTD3njBZEPXY/JK8Ydmn65LQ7un094mzZPD+6OJkN0kuxI4yN5OaqaUvWZGiiONHwRgt88DWQYm3UjSk5r2
D9L4zLIk6d70/pHmj/Y3KPf2eaT5K0yWk3RXCJ9bblxfuEX0nmPQXcn8CvMcdrjuZbL8JH9hIZdffJ60U8hBvk4w7estMwJUiuy0
u8Ecle79yjB75ZlfStIzw+o9MO0JOKL32oCFbSD3O2LwS+CZdNPue9odNr+YPve2K4aogEw7fWt72ZDPhuSVYojt3470ds1iTooZ
hmIWsr1FiklaFboVVufDT+/XZOH3BNPdGOqI6KD52X72QF6TvFrUPHo7kuVqm3w+i1go8tWPNFsOso8qTNdTRLNFPA2aEX96nzqM
DsjiccAVvV6TXmpm92ntSHptIPm1sFI0fbwmvTbMb1DEkUbkfjFGVPK6EbFRrpAicopoZHcWEl4YvG5kgyhfJauLLHpvF2rQHI2T
XomuU0SLs7fhp9ccXWiO5BHN+Xz2YerM66NtidlKu0dWH50VVUom0BCS3Ss35RXl4PVTVAmaGbPZfv6KdMeN8M6wvMHsU8iU8XHZ
Cxd0vvlx+LB9vVkrMmubnGpyK0iGk5RHcjjiafsYnGFgcwCTUc4JLpFyILk1Mq0xmLSicMphfcGvu377deHWJFNuaSCHHDEjJuMR
flAPYG4tmX70F5TUpfXZ8NM7NlsEfcWNU5NxCiafFJSGoBvJI3L7g2OSlpSXSScokhxMe9h9g2l0BtX9YBzGpdkx7bBY7C6/jpQq
KGbaQWVUv3uDcjI4uQGzvW2Q8CVDDIPloybLVmzyMOQJSBFLKYY3KKYgIX3F0FzYHOV6JE2TY5YjNImWsDCpfpw8qZ5EiqiW5Snk
WdCRX+URl1QTlc9FH02LVCPKjNjFZBrk2aTEHI/FmGrJM1RUWw6IAkdRjTwXDn49Flhpf3zbGx4QmuJoNRkOZYGWGwb5VelodizP
StIFikqC7X66LgX5PW44KeLT7qU8iUtPkF+mLDfGlYTGKapa5tiS4QtyLJY5JkIMdnALKUceuQUy6zR7wXRHSMrTYEq2QFlk2h3L
sbAht6KnsHBcS4a3omHT7kU3QE0RE1EPtMux+Oi6ILTqmO2FCDWEFesK6Svm9qXxRS6juPSlLRCT2cy/DyvD8yyXQu5WKZPUzLEl
xAaENwXLk1PYQjVojwY2f1of0HVkmGnIc4EQD4gpzyCczRmScJ38IiFxw0My6QYaVdgtMkQRYVrmtKNfICQv/CAbEBzJD3YFuV2N
syCE3C9Eu2LsEOaAALhXBgo5gjEISfKC0KT33Cv1LlflLjeCPYpHESPlIYGnAYK2znOgnlyusLnT2ZDbAxgFej5RBIITDMPWxvLu
BBgVmpwWCyks3nqRiz6o0qBWgqinlEAbFhDlIHiWoo3SENQFQSCY0QgPxdZLnogIkUPdmIgguXRJ5BDbHaaYA5phWJJDeaA0i5zC
KM2SQIrLUgVGvREWsyzNSiEBCEsKZrTJKQHNLzktrl1ycJopxY7rT4ZFTtcy7QzzGzKFYXsJT4x9MlkWMLyToHbsgcmh9ghKE3+i
9RSDAAQGKcMfRTykiKdYDs28FjGnjuXwnIEg/kkxn5liFuEtzaB2QgyodY4Ho/E5Mi5Sr8QLI3teEFS3boo5RTI+LSc5Bkt5Ltvf
pP5SuSLVMqOYbnLzRI5nY3qdZ7qaM0CEVzXnbhm7KQt203M8p1yRatjEvaLL89gI5YGyeE14jLEEafcE+XXjJ/0+rc64aTQ6GA8y
7U6IRVqTpzpxfol3J3ZVMYZowGPCmJw/iq6L5QIqzwXGMZGwujPseZKNVL6QvR/FXF+kcS7qzotBeQwtTzHfRXkqUNFEssjF6DuK
LJL5rhSZhJFFZBMjGmOA48DCLiP+SfyaO4c0PzJOweDyJnb02k2YhhYQnS8NgMQyGaadG+8yQFpTkGyp7CM1FAIQI+eoChJNM+cl
B4ouJXNxfynyApFwwMnfRC5Tpp1IMojOo8/1KLRoX2a6gtCoYyQIoUXKwwTbOebXKJewfjJPprBtOOJKNRna+VRv80z3kSRnWL0w
LK0Xiy0IGse8IlKmQkE9esXEHg5RhUEqzu8JaQvWnFX7hMlj0oigkILUgq/ND5BIkNew3OukPMERJOFmmfIUp4paKHA0S3sv5VQ1
DwiQJnpIkNdCykGpIjFitYGUTnByL2WZRC5x8o/Xy6BA+oZJHshrIfOpLIskegcZYmHML3EAqebAq4VIHIJkVsEjnisQS8pEgOpF
KWJ4Tk1SpmBYPYzFY+JniHqMmkmyk0LlcDnmWakeR9VCspsN1hzgJtLuaA7HU55GlV7FmWPGDLui4iPi7lCl2OWV6lGlWYIoKj68
Ei7BFnCZ17uQ4Gqq2Whm2cLLnBpFjvcEJWPeTfJzlOlpP/U8EDUNLNkKBjsw0zYnviMcVINnoIBpJtESUFUkJRMw5JHS2CIaa/4w
sdJiUrGl9gX1plgpFMmpJdLf8iSaROsY9caue6IeYbQ1QzClTCzxsoIXpKioYVi0CvMfSj7E6nuWKjAoLLBwq8kpA5k1qMlyGAFx
xVnhNLrUBRQ03GkS1EyKl5oIgNpDkFFXzKylzEcfNXAApQHJaTp2XdD4lgFdVvChghPaKdkilToVK9Qmpy6tL6gxG5HewDFQC0Yq
p42pNZmnKZv2yAQHfA7Uo9NN6yekmHTbpiYILYpymeCFXGouEn7cPDTQQyLPgWNIY/UgXTgNkYASZeCehWuW4npd1DN8KiUSyBfM
obGdF74g1DW3OmrAcG6SPSKr8kwzUsABalAA3nhFGF4QmLE8xUklfi9yq4+Spb0XhNGdnSL+gPYeGClMKRKVU6ydagAQVMK3TC+Y
XSnIywmjMnO1fQVVwT4Bg2JAlC5vb+EJpi0STM4uSYVFuUOM2m9EkSGZEdjxhV9Q3C8QlLO6rNQBcXPIrJaYu8EmGRDjxQogzUCY
5DVpb7Qb1QGH9JbarixOFTukknmVdwSDBfksK0dNZUNrC0Ehz/aWPDK1zCGLVirpLRE3DkYNBASSfU6MxeSbVRmJkpZTmicoGloe
TZlmQVFoG/kFKYrGJ8/JfgI6Eses8ACUbC7dbQl/SCIsFh5b8sYogsHOTxYrCE3weEl27VVRwR7RhlSqMjgVbQVlzham7V4VDQiW
J3hWFTVq7pUkytzuGchH4myVmKqfAxXKkF1nxApSHVCz7Dc53Y9XOYHCkRRTixdEezg31RkkjS56Y0a5OeUwyk1dByI9BBMtL2hA
kUGNMkAKiJJzpqz9wBYZFnKMTqarWAbG2iuIk+R9V8kpSgtFHc8zztEXvTM8uxbeFKprRs0RomB81Si/1TnjjKPmCoI6mjVXsBox
VfEcl17qeTNYwPwEViT4nFKDUecSVaAVp5QICskhg0HOy8STbwPl5Aqna1k4l0V9W/PslbJLZJJNXVcf56KBClFSle8GyqwlTvEK
gniJxHcB53qlK0L1qA+Y1gYM5qVQW+mkpDqFxrLfTBpVWKzlFksA1/C+hhQslC7c4bjKIYvKsecNRVQ5T3Lh7hSpZ8Ywvoxl9dYW
PLnm7la4om9iw/b1zh4KFh9GPSuimL1h/gYpZyeaNSa+UfKwPdHQH0ZY3b78z9vVdTzO8Je3183bkx/u40dN3oZfvD05ffVNc7Fo
7tbd9e3Nan3XXazW3W3zf//TrMPT94v4+ZnL+2590V10zeKq2RygQHgB+u1JuxntbnXXXcXBou1sfkPvjb/9t/CrJr63CVcu75fd
9r1fv7AxNd4OEi/RAYvt9aHjIt0wnuL2rr+/Ci9vbtarmzjucOfmgx+bl2/fG351vnoX7nq3vO2HTyd1hsfCL8OurBeXXXZHmMLm
hp+H4a9X7xfx4yHhtuv7q6v42+216VUKD5VVhvD50VV+++GvN/HTO6/v15+0SBS1RcY7dlqklRVR4qAwDy3yTfzd7+6XV13z1Z9v
gvLdftJaQ4by0FrpXNyOa8WQo35coOjnFSjY2bVWo64s0sqZbROrthnv2GmVBm1llc4dRm3lg2uVe1JbDxU/hOIAa42h40F3BOIz
hH2sVRhTkat0c2svVLUXdlylk7Ii0Zoj2nmVwtdWGe/Yzd0KWXO3s6MEXfW3eldPZLGySq+fiycyrhZAvT3MWvXsYEHUbBSkPMha
D+GPal53ZMp7B0YxtOBHF9nfsRswcjWBOjkv+hNY9bm7LtJprEkSjyrJzdd/dg4srhZY3NMPLA50DSTouUFCFciLXYG8hBqVoNST
T7IN1pgEd6C4+fBaxZ7iplQ1N6tmTrJRVQX6sST77cl3H37tmvPFeh2/BnJS9UZa1MgwnNsbzU8rSFUlw9zRxao/KtZvF2Fat6v4
Tej1/d1jJOurNOdhSIZHkJ1u58S7Cnft00+8TRU5OH/sZG1fjtiZKm0kDkMbPYgIAfdEG4GtWSriYSy1jgvVroy9qeFCOJAOP+CV
hkU86IbDaJfr7l2YURNm1oRpbD/uXvfIUtbIFnUYIHUA0Kiq5ScLc3tkuxOSepRHljWP7Gcusokqrvioe/oEXGFB1SwYnguNJqGG
jtWBaLQqL4G7ki8ea7S+hJnJF3Fwl6x9TcBOHUaZHxYw7kmZNdSqjlYcRpkfTAjQ7WmtCmuQSh8GZuCDjQHpE967Oilfk6s2Mye3
ZlfOYvHjh1/P769Wzftu3cXJV6NQLRdCmJl5E2anyPuoJEjYGol6mBRePpjcStiXxaqaFpvZOz/qaryzRF1NosIcuxtiX2mtqtZb
jX8u8cZDVa7+uXS5ANa6XEDM3Igmd+IpPiXpqZGpiHOHm/mrWVBzToBzC7ZOQJk9Fj9UFTlZfPrdhlpWqZjD9Pjgg11b6Vv+uxYC
qp2Vcu6u5/nrAGCrncCHYWBAzc4Xi1qHN0g1d32y2mwIuzYbClttHJ277Dw/wWR0tf9OHp0slbvapqoRSjBzMn4AozSmKskDsWYP
r1XtizWTNdO0Zm6MUHVAuKsDghrTACCPjul3tU2haodLpD46OJB7JspUtbhs3NMvxKGtNYAI9fTbmGyNI8O5g8sheiqrtJFWR+/V
2jWCjqDONISXR0+1d+0Bhmq/AxwGJhwgtmhfiy1OP5dCohO1DgC0z6Us4WpkJ6J5LmtFVbNXcSBewe5EAz7u4FstT/Nzh9KqAxZ2
vyS28LVuJSWe/mmFauOZd8+lPQl0ba0w+xnV+UOrUdUmO/Fc2AZZbWTXs1ck5ke/qto2acTxa/16529Z1DJvO3ciUz8mL/YbYnTV
Vq07+jHVXesvtvZpAJybLHtE79GesQNiDRvOfp7zEcT2zlmcqx51nNsxHeCYNVTd74G+mnSIvNxWJYpP/ty8NtUuZnj6NQpVOxIG
Rh8fNew7Z3M1XknBczmBYF31mwHiqGfDwh37ahh0tYPmeCCu5WFeSeyrKUeK6rk3/WzOvVU/G2Xg2AcPHqPDn1RqrIJ/o55+C7et
fQ4DAY/9zcl9HS0ZLWUa9avjo/6de8yq5+qlePo9ZrLa3asP5IUflCjui0bDag1ZmKf/iV9TPdDn5q5hyJ0IpkepbbVQo+VzUVtZ
7ZDU8FzWan0V7Msn38uC1cRcyKfflgTVD4QB/o0fCPpt50acryZv9rmc9hodMZ2GgOZvpSlg58+EiSrZYo7eerYrI6qgWmK0z+Vo
vNbVz8n753L0yVa/mAXuqAG19hncTyo3VuVrn0vXnaqifT17I7eevWguTfXDHeLYEFju7bBiDTdJeDZfaKx+Z1SaZ/BPI7jqB6z1
c/lqoam2sHt/7EY7tbdGu5pXAvdcKubWVru74Tg4YsbvgJnqv5fg566gCzE7l6ZE9SzYYUoXD58IQ7uv4ApVCvgw7ukR1L7buXBe
/Rd58Ol/naSKDA/05dgDVKScrX79WD2bUye6dsBaiONzpDtnb7XvHoOZm1qa/0szqnoK19hjL/Kj3b+f9CXC6gkxEM/lwInV1c7n
uQ+cHOCzi1g9Mi/00/iXNsN//v3t9c8nP/8/OW2v1JO+AAA=
"""

CARPETA = Path("datos")
CARPETA.mkdir(exist_ok=True)
archivos = json.loads(gzip.decompress(base64.b64decode(_PAQUETE)))
for nombre, contenido in archivos.items():
    (CARPETA / nombre).write_text(contenido, encoding="utf-8")
    print(f"  {nombre:<18} {contenido.count(chr(10)):>4} líneas")
print("Archivos listos en:", CARPETA.resolve())

  productos.csv        16 líneas
  pedidos.csv         135 líneas
  lineas_pedido.csv   392 líneas
  despachos.json     1305 líneas
Archivos listos en: C:\Users\Camila\Downloads\datos


In [3]:
# 0.3 · Funciones de apoyo (ejecútela; no la modifique)
_HUELLAS = {'n_dup_lineas': ['1b7d643272b21fa2'], 'n_vacios_despachada': ['f40ac1b8408d9bbe'], 'n_textos_ciudad': ['eb245c3a251f8473'], 'n_fechas_barra_entrega': ['3832b46c2106576b'], 'n_regla_2': ['03520a5922375593'], 'filas_lineas_1': ['ce235ff107ad2e44'], 'n_entrega_vacia': ['bd89df6fb3db8c2f'], 'n_retiradas_regla_2': ['08b620f38a5349fc'], 'lineas_cargadas': ['1a95e52e017a4039', 'dcb0c7a6afdfc678'], 'entregas_tarde': ['16c17111573c9323'], 'peor_proveedor': ['379f488fab548507'], 'peor_transportadora': ['44a6e6308bba3733'], 'peor_ciudad': ['8bfb473b9a7c421e'], 'peor_transportadora_ciudad': ['ce4736ef12d0c50c']}
_AUTOVERIFICACION = {}
CIUDADES_OK = {"Neiva", "Pitalito", "Garzón", "La Plata", "Campoalegre"}
CATEGORIAS_OK = {"Abarrotes", "Bebidas", "Aseo", "Lácteos"}

def _normalizar(v):
    if isinstance(v, (tuple, list)):
        return "(" + ",".join(_normalizar(x) for x in v) + ")"
    try:
        f = float(v)
        return str(int(f)) if f.is_integer() else f"{f:.2f}"
    except (TypeError, ValueError):
        return str(v).strip()

def _huella(clave, valor):
    return hashlib.sha256(f"CD-2026B-C2-G|{clave}|{_normalizar(valor)}".encode("utf-8")).hexdigest()[:16]

def verificar(clave, descripcion, valor, pista="Revise el cambio 👉 de esta celda."):
    """Compara un resultado con la cifra de control SIN mostrarla. Solo avisa: nunca detiene el cuaderno."""
    try:
        ok = _huella(clave, valor) in _HUELLAS[clave]
    except Exception:
        ok = False
    _AUTOVERIFICACION[clave] = ok
    if ok:
        print(f"✅ {descripcion}: coincide con la cifra de control.")
    else:
        print(f"⚠️ {descripcion}: todavía no coincide (su valor: {valor}). {pista}")

def comprobar(clave, descripcion, prueba):
    """Revisa una condición (por ejemplo, «solo quedan las 5 ciudades oficiales»). Solo avisa."""
    try:
        ok = bool(prueba())
    except Exception:
        ok = False
    _AUTOVERIFICACION[clave] = ok
    print(("✅ " if ok else "⚠️ ") + descripcion + ("" if ok else ": todavía no se cumple. Revise el cambio 👉."))

bitacora = []
def registrar(paso, tabla, filas_antes, filas_despues, decision):
    """Anota un paso de limpieza en la bitácora. Si la celda se repite, reemplaza el renglón (no lo duplica)."""
    bitacora[:] = [b for b in bitacora if b["paso"] != paso]
    bitacora.append({"paso": paso, "tabla": tabla, "filas_antes": int(filas_antes),
                     "filas_despues": int(filas_despues), "retiradas": int(filas_antes) - int(filas_despues),
                     "decision": decision})
    print(f"📒 {paso} · {tabla}: {filas_antes} → {filas_despues} filas")

def tratar_vacios(tabla, columna, opcion, mostrar=True):
    """Resuelve los vacíos de una columna según la opción elegida y devuelve una tabla NUEVA.
    Opciones: "conservar", "eliminar", "imputar_cero", "imputar_pedida" (rellena con cantidad_pedida)."""
    t = tabla.copy()
    vacios = t[columna].isna()
    if opcion == "eliminar":
        t = t[~vacios].copy()
    elif opcion == "imputar_cero":
        t[columna] = t[columna].fillna(0)
    elif opcion == "imputar_pedida" and "cantidad_pedida" in t.columns:
        t[columna] = t[columna].fillna(t["cantidad_pedida"])
    elif opcion != "conservar":
        print(f"⚠️ La opción «{opcion}» no existe aquí: se conservan los vacíos.")
    if pd.api.types.is_numeric_dtype(t[columna]) and t[columna].notna().all():
        t[columna] = t[columna].astype(int)
    if mostrar:
        print(f"{columna}: {int(vacios.sum())} vacíos antes → {int(t[columna].isna().sum())} después · "
              f"filas {len(tabla)} → {len(t)} (opción: {opcion})")
    return t

def es_fecha(serie):
    """True si la columna ya es de tipo fecha."""
    return pd.api.types.is_datetime64_any_dtype(serie)

comprobar("prueba", "La autoverificación funciona", lambda: 2 + 2 == 4)

✅ La autoverificación funciona


## Parte 1 · El caso: el CEDI de la Distribuidora La Ceiba
La **Distribuidora La Ceiba** (empresa ficticia) tiene un **centro de distribución (CEDI)** en Neiva que surte a **30 tiendas** de cinco municipios del Huila: Neiva, Pitalito, Garzón, La Plata y Campoalegre. Compra a **5 proveedores** y lleva los pedidos con **3 transportadoras**: su flota propia, TransHuila Express y Rápido Sur.

En **septiembre de 2026** las tiendas se quejaron de pedidos **incompletos** (llega menos de lo pedido) y **tardíos** (llegan después de la fecha prometida). La gerente de operaciones necesita tres respuestas, con datos:

> **1.** ¿Los pedidos incompletos se concentran en algún **proveedor**?
> **2.** ¿Las entregas tarde se deben a la **ciudad** o a la **transportadora**?
> **3.** ¿Qué **acción** se toma primero?

### 1.1 Las cuatro fuentes
| Archivo | De dónde sale | Qué trae (una fila = …) |
|---|---|---|
| `productos.csv` | Catálogo | un producto: código (`sku`), descripción, categoría y proveedor |
| `pedidos.csv` | Sistema de ventas | un pedido: código, fecha, tienda y ciudad |
| `lineas_pedido.csv` | Sistema de ventas | un producto dentro de un pedido: cuánto se pidió y cuánto salió del CEDI |
| `despachos.json` | API de las transportadoras | una guía de despacho: transportadora, fecha prometida, fecha real de entrega y novedad |

### 1.2 Reglas del negocio (con ellas se decide qué está mal)
- Un pedido tiene **varias líneas** (una por producto) y **una sola** guía de despacho.
- Ciudades válidas: **Neiva, Pitalito, Garzón, La Plata, Campoalegre**. Categorías válidas: **Abarrotes, Bebidas, Aseo, Lácteos**. Escritas exactamente así.
- Todas las fechas van en formato **AAAA-MM-DD** (por ejemplo, 2026-09-07).
- La cantidad despachada no puede ser negativa ni mayor que la cantidad pedida.
- **% despachado** = unidades despachadas ÷ unidades pedidas × 100. Mide qué tan completos salen los pedidos.
- Una entrega es **tarde** si la fecha de entrega es posterior a la fecha prometida (`fecha_compromiso`). Si no tiene fecha de entrega, todavía no se ha entregado: no cuenta ni a tiempo ni tarde.

### 1.3 Extraer: las fuentes tal como llegan (celda ya escrita)
**Extraer** es traer los datos **sin cambiarlos**. Los CSV se leen con `pd.read_csv`. El JSON de la API viene **anidado** (las dos fechas están guardadas dentro de un campo `fechas`); una sola línea, `pd.json_normalize`, lo **aplana** en una tabla común.

In [4]:
productos = pd.read_csv(CARPETA / "productos.csv")
pedidos = pd.read_csv(CARPETA / "pedidos.csv")
lineas = pd.read_csv(CARPETA / "lineas_pedido.csv")

with open(CARPETA / "despachos.json", encoding="utf-8") as f:
    respuesta_api = json.load(f)
print("Así llega un despacho desde la API:")
print(json.dumps(respuesta_api["despachos"][0], indent=2, ensure_ascii=False))
despachos = (pd.json_normalize(respuesta_api["despachos"])          # ← la línea que aplana el JSON
             .rename(columns={"fechas.compromiso": "fecha_compromiso", "fechas.entrega": "fecha_entrega"}))

print()
for nombre, df in [("productos", productos), ("pedidos", pedidos), ("lineas", lineas), ("despachos", despachos)]:
    print(f"{nombre:<10} {df.shape[0]:>4} filas · columnas: {', '.join(df.columns)}")

Así llega un despacho desde la API:
{
  "guia": "G-70189",
  "id_pedido": "PED-0027",
  "transportadora": "Flota propia",
  "fechas": {
    "compromiso": "2026-09-08",
    "entrega": "2026-09-07"
  },
  "novedad": null
}

productos    15 filas · columnas: sku, descripcion, categoria, proveedor
pedidos     134 filas · columnas: id_pedido, fecha_pedido, tienda, ciudad
lineas      391 filas · columnas: id_linea, id_pedido, sku, cantidad_pedida, cantidad_despachada
despachos   130 filas · columnas: guia, id_pedido, transportadora, novedad, fecha_compromiso, fecha_entrega


In [5]:
display(pedidos.head(6))
display(lineas.head(6))
display(despachos.head(6))
productos

,id_pedido,fecha_pedido,tienda,ciudad
0,PED-0130,2026-09-30,T-15,Pitalito
1,PED-0070,2026-09-16,T-07,Neiva
2,PED-0090,2026-09-21,T-07,Neiva
3,PED-0030,2026-09-07,T-04,neiva
4,PED-0073,2026-09-17,T-29,Campoalegre
5,PED-0064,2026-09-15,T-01,Neiva


,id_linea,id_pedido,sku,cantidad_pedida,cantidad_despachada
0,L-0277,PED-0095,SKU-1003,30,30.0
1,L-0220,PED-0076,SKU-1005,14,11.0
2,L-0382,PED-0129,SKU-1013,26,26.0
3,L-0106,PED-0038,SKU-1003,15,15.0
4,L-0286,PED-0098,SKU-1010,45,20.0
5,L-0385,PED-0130,SKU-1005,43,43.0


,guia,id_pedido,transportadora,novedad,fecha_compromiso,fecha_entrega
0,G-70189,PED-0027,Flota propia,NaN,2026-09-08,2026-09-07
1,G-70290,PED-0041,Rápido Sur,NaN,2026-09-12,2026-09-12
2,G-70739,PED-0105,TransHuila Express,NaN,2026-09-26,25/09/2026
3,G-70134,PED-0019,Rápido Sur,NaN,2026-09-07,2026-09-07
4,G-70515,PED-0073,Flota propia,NaN,2026-09-18,2026-09-18
5,G-70617,PED-0088,TransHuila Express,NaN,2026-09-23,23/09/2026


,sku,descripcion,categoria,proveedor
0,SKU-1001,Arroz blanco 1 kg,Abarrotes,Alimentos del Opia S.A.S.
1,SKU-1002,Aceite vegetal 1 L,Abarrotes,Alimentos del Opia S.A.S.
2,SKU-1003,Pasta spaghetti 500 g,Abarrotes,Alimentos del Opia S.A.S.
3,SKU-1004,Azúcar 1 kg,Abarrotes,Distribuciones Magdalena S.A.S.
4,SKU-1005,Café molido 500 g,Abarrotes,Distribuciones Magdalena S.A.S.
5,SKU-1006,"Gaseosa 1,5 L",Bebidas,Bebidas Río Claro Ltda.
6,SKU-1007,Agua 600 ml,BEBIDAS,Bebidas Río Claro Ltda.
7,SKU-1008,Jugo néctar 1 L,Bebidas,Bebidas Río Claro Ltda.
8,SKU-1009,Detergente en polvo 1 kg,aseo,Aseo Andino del Sur S.A.
9,SKU-1010,Lavaloza crema 500 g,Aseo,Aseo Andino del Sur S.A.


## Parte A · Diagnóstico: medir antes de corregir (1,0)
Como un médico que no receta sin examinar, un analista **primero mide** los problemas de los datos y **después** los corrige. Si no se mide el «antes», no se puede demostrar que se mejoró.

En cada pareja, ejecute el 📘 ejemplo, lea lo que muestra y luego haga el cambio 👉 en **su turno**. Todo se mide sobre las tablas **originales**. Cada conteo correcto vale 0,12.

### A.1 · Filas duplicadas
Una fila **duplicada** es la misma fila exportada dos veces. Hace daño porque **cuenta dos veces** lo mismo: un pedido repetido suma el doble de unidades. `.duplicated()` marca `True` en cada copia y `.sum()` cuenta los `True`.

In [6]:
# A.1 · 📘 Ejemplo resuelto: filas duplicadas en PEDIDOS
n_dup_pedidos = int(pedidos.duplicated().sum())
print("Pedidos duplicados:", n_dup_pedidos)
pedidos[pedidos.duplicated(keep=False)].sort_values("id_pedido")     # cada pedido repetido, junto a su copia

Pedidos duplicados: 4


,id_pedido,fecha_pedido,tienda,ciudad
113,PED-0019,2026-09-04,T-24,La Plata
129,PED-0019,2026-09-04,T-24,La Plata
93,PED-0054,2026-09-12,T-19,Garzón
105,PED-0054,2026-09-12,T-19,Garzón
2,PED-0090,2026-09-21,T-07,Neiva
8,PED-0090,2026-09-21,T-07,Neiva
26,PED-0112,2026-09-26,T-14,Pitalito
38,PED-0112,2026-09-26,T-14,Pitalito


In [7]:
# A.1 · 👉 Su turno: filas duplicadas en LÍNEAS
# 👉 Cambie `pedidos` por `lineas` en la línea siguiente (es el único cambio).
n_dup_lineas = int(lineas.duplicated().sum())
print("Líneas duplicadas:", n_dup_lineas)

verificar("n_dup_lineas", "A.1 · líneas duplicadas", n_dup_lineas)

Líneas duplicadas: 5
✅ A.1 · líneas duplicadas: coincide con la cifra de control.


### A.2 · Datos vacíos
Un dato **vacío** (`NaN`) es una casilla que nadie llenó. `.isna()` marca `True` donde falta el dato. Antes de decidir qué hacer con un vacío hay que mirar **por qué** falta.

In [8]:
# A.2 · 📘 Ejemplo resuelto: despachos SIN fecha de entrega
n_vacios_entrega = int(despachos["fecha_entrega"].isna().sum())
print("Despachos sin fecha de entrega:", n_vacios_entrega)
despachos[despachos["fecha_entrega"].isna()]          # mire la columna «novedad»

Despachos sin fecha de entrega: 3


,guia,id_pedido,transportadora,novedad,fecha_compromiso,fecha_entrega
28,G-70765,PED-0109,TransHuila Express,Reprogramado por la tienda,2026-09-28,NaN
34,G-70913,PED-0130,Rápido Sur,Reprogramado por la tienda,2026-10-02,NaN
117,G-70774,PED-0110,Rápido Sur,Reprogramado por la tienda,2026-09-28,NaN


In [9]:
# A.2 · 👉 Su turno: líneas SIN cantidad despachada
# 👉 Cambie `despachos` por `lineas` y "fecha_entrega" por "cantidad_despachada".
n_vacios_despachada = int(lineas["cantidad_despachada"].isna().sum())
print("Líneas sin cantidad despachada:", n_vacios_despachada)

verificar("n_vacios_despachada", "A.2 · líneas sin cantidad despachada", n_vacios_despachada)

Líneas sin cantidad despachada: 6
✅ A.2 · líneas sin cantidad despachada: coincide con la cifra de control.


### A.3 · Textos escritos de varias formas
Para una persona, `Neiva`, `NEIVA` y `Neiva ` (con un espacio al final) son la misma ciudad; para el computador son **tres textos distintos**, y un conteo por ciudad saldría partido en tres. `.unique()` lista los textos distintos (las comillas dejan ver los espacios escondidos) y `.nunique()` los cuenta.

In [10]:
# A.3 · 📘 Ejemplo resuelto: textos distintos de CATEGORÍA en productos
print(sorted(productos["categoria"].unique()))
n_textos_categoria = int(productos["categoria"].nunique())
print("Textos distintos de categoría:", n_textos_categoria, "(deberían ser 4)")

['Abarrotes', 'Aseo', 'BEBIDAS', 'Bebidas', 'Lacteos', 'Lácteos', 'aseo']
Textos distintos de categoría: 7 (deberían ser 4)


In [11]:
# A.3 · 👉 Su turno: textos distintos de CIUDAD en pedidos
# 👉 En las DOS líneas siguientes, cambie `productos` por `pedidos` y "categoria" por "ciudad".
print(sorted(pedidos["ciudad"].unique()))
n_textos_ciudad = int(pedidos["ciudad"].nunique())
print("Textos distintos de ciudad:", n_textos_ciudad, "(deberían ser 5)")

verificar("n_textos_ciudad", "A.3 · textos distintos de ciudad", n_textos_ciudad)

['Campo Alegre', 'Campoalegre', 'GARZÓN', 'Garzon', 'Garzón', 'LA PLATA', 'La Plata', 'La plata', 'Neiva', 'PITALITO', 'Pitalito', 'Pitalito ', 'campoalegre', 'garzón', 'la plata', 'neiva', 'pitalito']
Textos distintos de ciudad: 17 (deberían ser 5)
✅ A.3 · textos distintos de ciudad: coincide con la cifra de control.


### A.4 · Fechas en dos formatos
La regla dice **AAAA-MM-DD**. Una barra `/` delata que alguien escribió **DD/MM/AAAA**. El peligro: `07/09/2026` puede leerse como 7 de septiembre (costumbre colombiana) o como 9 de julio (costumbre de Estados Unidos), y el computador **no avisa** si se equivoca. `.str.contains("/")` marca `True` en las fechas que tienen barra.

In [12]:
# A.4 · 📘 Ejemplo resuelto: fechas de PEDIDO escritas con barra
con_barra = pedidos["fecha_pedido"].astype(str).str.contains("/", regex=False)
n_fechas_barra_pedido = int(con_barra.sum())
print("Fechas de pedido en formato DD/MM/AAAA:", n_fechas_barra_pedido)
pedidos.loc[con_barra, "fecha_pedido"].head()

Fechas de pedido en formato DD/MM/AAAA: 16


15    01/09/2026
18    29/09/2026
57    15/09/2026
63    01/09/2026
66    25/09/2026
Name: fecha_pedido, dtype: str

In [13]:
# A.4 · 👉 Su turno: fechas de ENTREGA escritas con barra (tabla despachos)
# 👉 En la línea siguiente, cambie `pedidos` por `despachos` y "fecha_pedido" por "fecha_entrega".
con_barra = despachos["fecha_entrega"].astype(str).str.contains("/", regex=False)
n_fechas_barra_entrega = int(con_barra.sum())
print("Fechas de entrega en formato DD/MM/AAAA:", n_fechas_barra_entrega)

verificar("n_fechas_barra_entrega", "A.4 · fechas de entrega en DD/MM/AAAA", n_fechas_barra_entrega)

Fechas de entrega en formato DD/MM/AAAA: 40
✅ A.4 · fechas de entrega en DD/MM/AAAA: coincide con la cifra de control.


### A.5 · Valores imposibles
Un valor es **imposible** cuando viola una regla del negocio. Aquí hay dos reglas para la cantidad despachada:
- **Regla 1:** no puede ser **negativa**.
- **Regla 2:** no puede ser **mayor que la cantidad pedida** (no se despacha más de lo que la tienda pidió).

Cada regla es una condición que da `True` o `False` en cada fila; `.sum()` cuenta las filas que la violan.

In [14]:
# A.5 · 📘 Ejemplo resuelto: regla 1, cantidad despachada NEGATIVA
regla_1 = lineas["cantidad_despachada"] < 0
n_regla_1 = int(regla_1.sum())
print("Líneas con cantidad despachada negativa:", n_regla_1)
lineas[regla_1]

Líneas con cantidad despachada negativa: 2


,id_linea,id_pedido,sku,cantidad_pedida,cantidad_despachada
205,L-0129,PED-0046,SKU-1008,41,-41.0
345,L-0032,PED-0012,SKU-1004,41,-41.0


In [15]:
# A.5 · 👉 Su turno: regla 2, cantidad despachada MAYOR QUE la pedida
# 👉 Cambie la condición de la línea siguiente por la regla 2.
#    Pista: compare lineas["cantidad_despachada"] con lineas["cantidad_pedida"] usando el signo >
regla_2 = lineas["cantidad_despachada"] > lineas["cantidad_pedida"]
n_regla_2 = int(regla_2.sum())
print("Líneas que despachan más de lo pedido:", n_regla_2)
display(lineas[regla_2])

verificar("n_regla_2", "A.5 · líneas que despachan más de lo pedido", n_regla_2)

Líneas que despachan más de lo pedido: 3


,id_linea,id_pedido,sku,cantidad_pedida,cantidad_despachada
212,L-0055,PED-0020,SKU-1013,32,320.0
214,L-0180,PED-0062,SKU-1007,38,380.0
390,L-0105,PED-0038,SKU-1014,28,280.0


✅ A.5 · líneas que despachan más de lo pedido: coincide con la cifra de control.


### A.6 · El reporte «antes» (celda ya escrita)
Reúne los conteos de los ejemplos y de su turno en una sola tabla. Al final de la Parte B se le agrega la columna «después».

In [16]:
reporte = pd.DataFrame([
    ("Unicidad",     "pedidos duplicados",                          n_dup_pedidos),
    ("Unicidad",     "líneas duplicadas",                           n_dup_lineas),
    ("Completitud",  "despachos sin fecha de entrega",              n_vacios_entrega),
    ("Completitud",  "líneas sin cantidad despachada",              n_vacios_despachada),
    ("Consistencia", "textos distintos de categoría (deben ser 4)", n_textos_categoria),
    ("Consistencia", "textos distintos de ciudad (deben ser 5)",    n_textos_ciudad),
    ("Formato",      "fechas de pedido en DD/MM/AAAA",              n_fechas_barra_pedido),
    ("Formato",      "fechas de entrega en DD/MM/AAAA",             n_fechas_barra_entrega),
    ("Validez",      "líneas con despachado negativo",              n_regla_1),
    ("Validez",      "líneas que despachan más de lo pedido",       n_regla_2),
], columns=["dimension", "problema", "antes"])
reporte

,dimension,problema,antes
0,Unicidad,pedidos duplicados,4
1,Unicidad,líneas duplicadas,5
2,Completitud,despachos sin fecha de entrega,3
3,Completitud,líneas sin cantidad despachada,6
4,Consistencia,textos distintos de categoría (deben ser 4),7
5,Consistencia,textos distintos de ciudad (deben ser 5),17
6,Formato,fechas de pedido en DD/MM/AAAA,16
7,Formato,fechas de entrega en DD/MM/AAAA,40
8,Validez,líneas con despachado negativo,2
9,Validez,líneas que despachan más de lo pedido,3


🧠 **A.7 · ¿Qué problema pesa más? (0,4)**
- **(a)** De los problemas del reporte, ¿cuál cambiaría **más** la respuesta a la pregunta 1 de la gerente (pedidos incompletos por proveedor) si no se corrigiera? Explíquelo con una fila concreta de los datos (por ejemplo, una de las que mostró A.5). *(0,2)*
- **(b)** Los despachos sin fecha de entrega que mostró el ejemplo de A.2: ¿son un **error** o un **dato válido**? Mire su columna `novedad` y diga qué haría con ellos al medir las entregas tarde. *(0,1)*
- **(c)** Si una fecha de entrega `07/09/2026` se leyera como 9 de julio, ¿qué le pasaría a la medición de entregas tarde? *(0,1)*

**A.7.** (a) El problema que más puede cambiar la conclusión sobre los proveedores son los valores imposibles de cantidad despachada. Por ejemplo, la línea L-0055 tiene 32 unidades pedidas y 320 despachadas; ese dato exageraría artificialmente el cumplimiento del proveedor si no se retira. También los duplicados pueden contar dos veces unidades. (b) Los tres despachos sin fecha se deben conservar cuando la novedad indica que el pedido fue reprogramado o sigue sin confirmación: no prueban que haya llegado tarde ni a tiempo. Se excluyen del cálculo de tardanza hasta tener fecha confirmada y se reportan para seguimiento. (c) Leer 07/09/2026 como 9 de julio cambia el orden temporal y puede clasificar erróneamente una entrega como tardía o puntual, alterando el porcentaje.

## Parte B · Limpieza paso a paso (1,5)
Tres reglas que ya se usaron en los talleres:
1. **El original no se toca.** Cada paso crea una tabla **nueva** a partir de la anterior. Así cualquier celda se puede repetir sin dañar nada.
2. **Un problema por paso**, en este orden: primero lo que cambia la identidad de las filas (duplicados, textos, fechas) y al final lo que las elimina (vacíos e imposibles).
3. **Todo queda anotado** en la **bitácora**: cada celda ya trae su línea `registrar(...)`.

El recorrido de las tablas es este (la última de cada fila es la tabla limpia):

```
pedidos   → pedidos_1 (B.1) → pedidos_2 (B.2) → pedidos_limpio (B.3)
productos → productos_limpio (B.2)
despachos → despachos_1 (B.3) → despachos_limpio (B.4)
lineas    → lineas_1 (B.1) → lineas_2 (B.4) → lineas_3 (B.5) → lineas_limpio (B.5)
```
Cada paso de **su turno** vale 0,2.

### B.1 · Quitar los duplicados
`.drop_duplicates()` deja **una sola** copia de cada fila repetida.

In [17]:
# B.1 · 📘 Ejemplo resuelto: duplicados de PEDIDOS
pedidos_1 = pedidos.drop_duplicates()
registrar("B.1a Quitar duplicados", "pedidos", len(pedidos), len(pedidos_1),
          "Eliminar: es el mismo pedido exportado dos veces")

📒 B.1a Quitar duplicados · pedidos: 134 → 130 filas


In [18]:
# B.1 · 👉 Su turno: duplicados de LÍNEAS
# 👉 Reemplace .copy() por .drop_duplicates(), como en el ejemplo.
lineas_1 = lineas.drop_duplicates()
registrar("B.1b Quitar duplicados", "lineas", len(lineas), len(lineas_1),
          "Eliminar: es la misma línea exportada dos veces")

verificar("filas_lineas_1", "B.1 · filas de lineas_1", len(lineas_1))

📒 B.1b Quitar duplicados · lineas: 391 → 386 filas
✅ B.1 · filas de lineas_1: coincide con la cifra de control.


### B.2 · Homologar los textos
**Homologar** es llevar todas las formas de escribir algo a **una sola forma oficial**. Se hace en dos movimientos:
1. `.str.strip()` quita los espacios del inicio y del final; luego se unifican las mayúsculas. `.str.capitalize()` deja en mayúscula solo la primera letra (`BEBIDAS` → `Bebidas`); `.str.title()` pone en mayúscula la primera letra de **cada palabra** (`LA PLATA` → `La Plata`), que es lo que necesitan los nombres de ciudad.
2. Un **diccionario** `{"forma mal escrita": "forma oficial"}` con `.replace(...)` arregla lo que las mayúsculas no arreglan: una tilde que falta o una palabra partida en dos.

In [19]:
# B.2 · 📘 Ejemplo resuelto: CATEGORÍA de productos
productos_limpio = productos.copy()
productos_limpio["categoria"] = productos_limpio["categoria"].astype(str).str.strip().str.capitalize()
productos_limpio["categoria"] = productos_limpio["categoria"].replace({"Lacteos": "Lácteos"})    # recupera la tilde
print(productos_limpio["categoria"].value_counts().to_dict())
registrar("B.2a Homologar categoría", "productos", len(productos), len(productos_limpio),
          "strip + capitalize + diccionario (Lacteos → Lácteos): 4 categorías oficiales")
comprobar("b2_categoria", "B.2 · quedan solo las 4 categorías oficiales",
          lambda: set(productos_limpio["categoria"]) == CATEGORIAS_OK)

{'Abarrotes': 5, 'Aseo': 4, 'Bebidas': 3, 'Lácteos': 3}
📒 B.2a Homologar categoría · productos: 15 → 15 filas
✅ B.2 · quedan solo las 4 categorías oficiales


In [20]:
# B.2 · 👉 Su turno: CIUDAD de pedidos (dos cambios pequeños)
pedidos_2 = pedidos_1.copy()
# 👉 Cambio 1: agregue .str.title() al final de la línea siguiente (después de .str.strip()).
pedidos_2["ciudad"] = pedidos_2["ciudad"].astype(str).str.strip().str.title().str.title()
# 👉 Cambio 2: ejecute una vez, mire la lista que se imprime y agregue al diccionario la pareja
#    que corrige la ciudad que todavía sale escrita en DOS palabras.
pedidos_2["ciudad"] = pedidos_2["ciudad"].replace({"Garzon": "Garzón", "Campo Alegre": "Campoalegre"})
print(pedidos_2["ciudad"].value_counts().to_dict())
registrar("B.2b Homologar ciudad", "pedidos", len(pedidos_1), len(pedidos_2),
          "strip + title + diccionario: 5 ciudades oficiales")
comprobar("b2_ciudad", "B.2 · quedan solo las 5 ciudades oficiales", lambda: set(pedidos_2["ciudad"]) == CIUDADES_OK)

{'Pitalito': 32, 'Garzón': 30, 'Neiva': 29, 'La Plata': 24, 'Campoalegre': 15}
📒 B.2b Homologar ciudad · pedidos: 130 → 130 filas
✅ B.2 · quedan solo las 5 ciudades oficiales


### B.3 · Fechas a un solo formato
No se le deja al computador «adivinar» el formato. Se lee la columna **dos veces**, una con cada formato, y se unen las dos lecturas:
- `format="%Y-%m-%d"` entiende `2026-09-07`; `format="%d/%m/%Y"` entiende `07/09/2026`.
- `errors="coerce"` deja **vacío** (`NaT`) lo que no entiende, en lugar de detener el cuaderno.
- `iso.fillna(dmy)` toma la primera lectura y llena sus huecos con la segunda.

> ⚠️ No use el atajo `pd.to_datetime(..., format="mixed", dayfirst=True)`: en pandas 3 voltea en silencio las fechas AAAA-MM-DD (taller de la semana 9).

In [21]:
# B.3 · 📘 Ejemplo resuelto: FECHA DE PEDIDO
pedidos_limpio = pedidos_2.copy()
iso = pd.to_datetime(pedidos_limpio["fecha_pedido"], format="%Y-%m-%d", errors="coerce")    # entiende 2026-09-07
dmy = pd.to_datetime(pedidos_limpio["fecha_pedido"], format="%d/%m/%Y", errors="coerce")    # entiende 07/09/2026
pedidos_limpio["fecha_pedido"] = iso.fillna(dmy)                                            # une las dos lecturas
print("Leídas como AAAA-MM-DD:", int(iso.notna().sum()), "· leídas como DD/MM/AAAA:", int(dmy.notna().sum()))
print("Fechas que quedaron vacías:", int(pedidos_limpio["fecha_pedido"].isna().sum()))
print("Desde", pedidos_limpio["fecha_pedido"].min().date(), "hasta", pedidos_limpio["fecha_pedido"].max().date())
registrar("B.3a Fechas de pedido", "pedidos", len(pedidos_2), len(pedidos_limpio),
          "Formato por formato (AAAA-MM-DD y DD/MM/AAAA) y unir con fillna")

Leídas como AAAA-MM-DD: 114 · leídas como DD/MM/AAAA: 16
Fechas que quedaron vacías: 0
Desde 2026-09-01 hasta 2026-09-30
📒 B.3a Fechas de pedido · pedidos: 130 → 130 filas


In [22]:
# B.3 · 👉 Su turno: FECHA DE ENTREGA de los despachos
despachos_1 = despachos.copy()
despachos_1["fecha_compromiso"] = pd.to_datetime(despachos_1["fecha_compromiso"], format="%Y-%m-%d",
                                                 errors="coerce")       # (ya escrita) viene toda en AAAA-MM-DD
iso = pd.to_datetime(despachos_1["fecha_entrega"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(despachos_1["fecha_entrega"], format="%d/%m/%Y", errors="coerce")   # formato DD/MM/AAAA
despachos_1["fecha_entrega"] = iso.fillna(dmy)
n_entrega_vacia = int(despachos_1["fecha_entrega"].isna().sum())
print("Fechas de entrega vacías después de convertir:", n_entrega_vacia)
registrar("B.3b Fechas de entrega", "despachos", len(despachos), len(despachos_1),
          "Formato por formato (AAAA-MM-DD y DD/MM/AAAA) y unir con fillna")

verificar("n_entrega_vacia", "B.3 · fechas de entrega vacías después de convertir", n_entrega_vacia)

Fechas de entrega vacías después de convertir: 3
📒 B.3b Fechas de entrega · despachos: 130 → 130 filas
✅ B.3 · fechas de entrega vacías después de convertir: coincide con la cifra de control.


### B.4 · Decidir qué hacer con los vacíos
Frente a un dato que falta hay varios caminos, y escoger es una **decisión de ingeniero**, no de programación. La función `tratar_vacios` (definida en la celda 0.3) ofrece cuatro opciones:

| Opción | Qué hace | Riesgo |
|---|---|---|
| `"conservar"` | Deja el vacío como está | Los cálculos lo ignoran o lo cuentan mal |
| `"eliminar"` | Quita las filas con el vacío | Se pierden esas filas |
| `"imputar_cero"` | Rellena con 0 | Supone que no salió nada del CEDI |
| `"imputar_pedida"` | Rellena con la cantidad pedida | Supone que salió todo lo pedido |

In [23]:
# B.4 · 📘 Ejemplo resuelto: despachos sin FECHA DE ENTREGA → se CONSERVAN
despachos_limpio = tratar_vacios(despachos_1, "fecha_entrega", "conservar")
registrar("B.4a Vacíos en fecha_entrega", "despachos", len(despachos_1), len(despachos_limpio),
          "Conservar: el pedido existe y fue reprogramado por la tienda; no se inventa una fecha")

fecha_entrega: 3 vacíos antes → 3 después · filas 130 → 130 (opción: conservar)
📒 B.4a Vacíos en fecha_entrega · despachos: 130 → 130 filas


Antes de su turno, esta celda (ya escrita) muestra **qué le pasa al indicador** con cada opción, en todo el CEDI y solo en la categoría **Aseo**, donde está la mayoría de los vacíos. Las líneas imposibles se dejan por fuera para que no distorsionen la comparación (se tratan en B.5).

In [24]:
# B.4 · Efecto de cada opción sobre el % despachado (ejecútela y compare)
_posibles = lineas_1[~((lineas_1["cantidad_despachada"] < 0) |
                       (lineas_1["cantidad_despachada"] > lineas_1["cantidad_pedida"]))]
_aseo = set(productos_limpio.loc[productos_limpio["categoria"] == "Aseo", "sku"])    # productos de la categoría Aseo
_filas = []
for _op in ["conservar", "eliminar", "imputar_cero", "imputar_pedida"]:
    _t = tratar_vacios(_posibles, "cantidad_despachada", _op, mostrar=False)
    _a = _t[_t["sku"].isin(_aseo)]
    _filas.append({"opción": _op, "líneas": len(_t),
                   "vacíos que quedan": int(_t["cantidad_despachada"].isna().sum()),
                   "% despachado (todo)": round(100 * _t["cantidad_despachada"].sum() / _t["cantidad_pedida"].sum(), 2),
                   "% despachado (Aseo)": round(100 * _a["cantidad_despachada"].sum() / _a["cantidad_pedida"].sum(), 2)})
opciones_vacios = pd.DataFrame(_filas)
opciones_vacios

,opción,líneas,vacíos que quedan,% despachado (todo),% despachado (Aseo)
0,conservar,381,6,96.01,87.72
1,eliminar,375,0,96.93,89.91
2,imputar_cero,381,0,96.01,87.72
3,imputar_pedida,381,0,96.96,90.15


In [25]:
# B.4 · 👉 Su turno: líneas sin CANTIDAD DESPACHADA
# 👉 Cambie "conservar" por la opción que usted elija: "eliminar", "imputar_cero" o "imputar_pedida".
#    Su decisión se justifica en la pregunta B.7.
OPCION = "imputar_cero"
lineas_2 = tratar_vacios(lineas_1, "cantidad_despachada", OPCION)
registrar("B.4b Vacíos en cantidad_despachada", "lineas", len(lineas_1), len(lineas_2),
          f"Opción elegida: {OPCION} (justificación en B.7)")
comprobar("b4_vacios", "B.4 · no quedan vacíos en cantidad_despachada",
          lambda: lineas_2["cantidad_despachada"].notna().all())

cantidad_despachada: 6 vacíos antes → 0 después · filas 386 → 386 (opción: imputar_cero)
📒 B.4b Vacíos en cantidad_despachada · lineas: 386 → 386 filas
✅ B.4 · no quedan vacíos en cantidad_despachada


### B.5 · Retirar los valores imposibles
Se podría intentar «corregirlos» (quitar el cero de más, quitar el signo), pero **adivinar la corrección es inventar datos**. Lo correcto es retirarlos del análisis, anotarlo en la bitácora y reportarlos al área que los registró. El símbolo `~` significa **«no»**: `tabla[~regla]` deja las filas que **no** violan la regla.

In [26]:
# B.5 · 📘 Ejemplo resuelto: regla 1 (despachado negativo)
regla_1 = lineas_2["cantidad_despachada"] < 0
lineas_3 = lineas_2[~regla_1].copy()
registrar("B.5a Despachado negativo", "lineas", len(lineas_2), len(lineas_3),
          "Eliminar: el valor real se desconoce; no se adivina. Se reporta a bodega")

📒 B.5a Despachado negativo · lineas: 386 → 384 filas


In [27]:
# B.5 · 👉 Su turno: regla 2 (despachado mayor que lo pedido)
# 👉 Cambie la condición por la regla 2, la misma de A.5, pero ahora sobre lineas_3.
regla_2 = lineas_3["cantidad_despachada"] > lineas_3["cantidad_pedida"]
lineas_limpio = lineas_3[~regla_2].copy()
n_retiradas_regla_2 = len(lineas_3) - len(lineas_limpio)
registrar("B.5b Despachado mayor que lo pedido", "lineas", len(lineas_3), len(lineas_limpio),
          "Eliminar: el valor real se desconoce; no se adivina. Se reporta a bodega")

verificar("n_retiradas_regla_2", "B.5 · líneas retiradas por la regla 2", n_retiradas_regla_2)

📒 B.5b Despachado mayor que lo pedido · lineas: 384 → 381 filas
✅ B.5 · líneas retiradas por la regla 2: coincide con la cifra de control.


### B.6 · Bitácora y reporte «antes / después» (celda ya escrita)
La columna `despues` se mide sobre **sus** tablas limpias. Un valor **no** tiene que llegar a cero: los despachos sin fecha de entrega, que se conservaron a propósito.

In [28]:
def _medir(funcion):
    try:
        return int(funcion())
    except Exception:
        return None

def _con_barra(serie):
    return 0 if es_fecha(serie) else int(serie.astype(str).str.contains("/", regex=False).sum())

reporte["despues"] = [
    _medir(lambda: pedidos_limpio.duplicated().sum()),
    _medir(lambda: lineas_limpio.duplicated().sum()),
    _medir(lambda: despachos_limpio["fecha_entrega"].isna().sum()),
    _medir(lambda: lineas_limpio["cantidad_despachada"].isna().sum()),
    _medir(lambda: productos_limpio["categoria"].nunique()),
    _medir(lambda: pedidos_limpio["ciudad"].nunique()),
    _medir(lambda: _con_barra(pedidos_limpio["fecha_pedido"])),
    _medir(lambda: _con_barra(despachos_limpio["fecha_entrega"])),
    _medir(lambda: (lineas_limpio["cantidad_despachada"] < 0).sum()),
    _medir(lambda: (lineas_limpio["cantidad_despachada"] > lineas_limpio["cantidad_pedida"]).sum()),
]
print("BITÁCORA DE LIMPIEZA")
display(pd.DataFrame(bitacora))
print("REPORTE ANTES / DESPUÉS")
reporte

BITÁCORA DE LIMPIEZA


,paso,tabla,filas_antes,filas_despues,retiradas,decision
0,B.1a Quitar duplicados,pedidos,134,130,4,Eliminar: es el mismo pedido exportado dos veces
1,B.1b Quitar duplicados,lineas,391,386,5,Eliminar: es la misma línea exportada dos veces
2,B.2a Homologar categoría,productos,15,15,0,strip + capitalize + diccionario (Lacteos → Lá...
3,B.2b Homologar ciudad,pedidos,130,130,0,strip + title + diccionario: 5 ciudades oficiales
4,B.3a Fechas de pedido,pedidos,130,130,0,Formato por formato (AAAA-MM-DD y DD/MM/AAAA) ...
5,B.3b Fechas de entrega,despachos,130,130,0,Formato por formato (AAAA-MM-DD y DD/MM/AAAA) ...
6,B.4a Vacíos en fecha_entrega,despachos,130,130,0,Conservar: el pedido existe y fue reprogramado...
7,B.4b Vacíos en cantidad_despachada,lineas,386,386,0,Opción elegida: imputar_cero (justificación en...
8,B.5a Despachado negativo,lineas,386,384,2,Eliminar: el valor real se desconoce; no se ad...
9,B.5b Despachado mayor que lo pedido,lineas,384,381,3,Eliminar: el valor real se desconoce; no se ad...


REPORTE ANTES / DESPUÉS


,dimension,problema,antes,despues
0,Unicidad,pedidos duplicados,4,0
1,Unicidad,líneas duplicadas,5,0
2,Completitud,despachos sin fecha de entrega,3,3
3,Completitud,líneas sin cantidad despachada,6,0
4,Consistencia,textos distintos de categoría (deben ser 4),7,4
5,Consistencia,textos distintos de ciudad (deben ser 5),17,5
6,Formato,fechas de pedido en DD/MM/AAAA,16,0
7,Formato,fechas de entrega en DD/MM/AAAA,40,0
8,Validez,líneas con despachado negativo,2,0
9,Validez,líneas que despachan más de lo pedido,3,0


🧠 **B.7 · Las decisiones de limpieza (0,5)**
- **(a)** ¿Qué opción eligió en B.4 para las líneas sin cantidad despachada y por qué? Use la tabla «efecto de cada opción» y piense en **qué mide** el indicador: ¿qué opción lo infla, cuál lo castiga y cuál no inventa nada? *(0,2)*
- **(b)** ¿Por qué las líneas imposibles se retiran en lugar de «arreglarlas» (por ejemplo, quitando el cero de más)? ¿A qué área de la empresa le reportaría esas líneas? *(0,15)*
- **(c)** Los despachos sin fecha de entrega se **conservaron**, pero a las líneas sin cantidad despachada se les aplicó otra decisión. ¿Por qué un mismo tipo de problema (un dato vacío) puede llevar a decisiones distintas? *(0,15)*

**B.7.** (a) Elegí `imputar_cero` para las seis cantidades faltantes. El indicador mide unidades despachadas frente a unidades pedidas; asignar cero es una decisión conservadora porque no atribuye unidades que no están registradas. En la tabla, el porcentaje global queda en 96,01%; eliminar las filas lo eleva a 96,93% y puede ocultar faltantes, mientras imputar lo pedido lo eleva a 96,96% y supone que sí se entregó todo. (b) Las cantidades imposibles se retiran porque no se conoce el valor real y corregirlas a ojo inventaría datos. Se reportan al área de Bodega/Despachos para revisar el registro y el proceso. (c) Una fecha vacía puede representar un despacho reprogramado todavía válido, por eso se conserva y se excluye del cálculo de tardanza; una cantidad vacía afecta directamente el indicador de unidades y se debe tratar explícitamente para no dejar el cálculo incompleto.

## Parte C · Integrar, cargar y consultar con SQL (1,2)
### C.0 · Integrar y cargar (celda ya escrita)
Ahora se juntan las tablas limpias. `merge` hace lo mismo que **BUSCARV** en Excel o un `JOIN` en SQL: a cada línea le pega los datos de **su** producto (categoría y proveedor) y de **su** pedido (tienda y ciudad); a cada despacho le pega la ciudad de su pedido. `validate=` revisa que la relación sea la esperada (muchas líneas → un producto; un despacho ↔ un pedido) y se detiene si no lo es.

La celda calcula además el **estado** de cada entrega (`a tiempo`, `tarde` o `sin fecha`) y carga dos tablas en la base **SQLite** `cedi.db`: `lineas` y `entregas`. Ejecútela y lea el resumen.

In [29]:
def _usar(nombre, respaldo):
    """Toma la tabla limpia; si no existe (se saltó una celda), usa la original para no detener el cuaderno."""
    if nombre in globals():
        return globals()[nombre]
    print(f"⚠️ No existe «{nombre}»: ¿se saltó una celda? Se usa la tabla original.")
    return respaldo

def _a_fecha(serie):
    return serie if es_fecha(serie) else pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")

try:
    lineas_detalle = (_usar("lineas_limpio", lineas)
                      .merge(_usar("productos_limpio", productos), on="sku", how="left", validate="many_to_one")
                      .merge(_usar("pedidos_limpio", pedidos), on="id_pedido", how="left", validate="many_to_one"))
    entregas = _usar("despachos_limpio", despachos).merge(_usar("pedidos_limpio", pedidos)[["id_pedido", "tienda", "ciudad"]],
                                                          on="id_pedido", how="left", validate="one_to_one")
    _fc, _fe = _a_fecha(entregas["fecha_compromiso"]), _a_fecha(entregas["fecha_entrega"])
    entregas["estado"] = np.where(_fe.isna(), "sin fecha", np.where(_fe > _fc, "tarde", "a tiempo"))
    entregas["dias_retraso"] = (_fe - _fc).dt.days.clip(lower=0)
except Exception as error:
    print(f"⚠️ La integración no se pudo hacer: {type(error).__name__}: {error}")
    print("   Revise la limpieza (Parte B) y vuelva a ejecutar esta celda.")

def _a_texto(df):
    df = df.copy()
    for c in df.columns:
        if es_fecha(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d")       # SQLite guarda las fechas como texto AAAA-MM-DD
    return df

try:
    con.close()                                         # si la celda se repite, se cierra la conexión anterior
except Exception:
    pass
con = sqlite3.connect("cedi.db")
try:
    _a_texto(lineas_detalle).to_sql("lineas", con, if_exists="replace", index=False)
    _a_texto(entregas).to_sql("entregas", con, if_exists="replace", index=False)
    print(f"Tabla «lineas»:   {len(lineas_detalle)} filas · {', '.join(lineas_detalle.columns)}")
    print(f"Tabla «entregas»: {len(entregas)} filas · {', '.join(entregas.columns)}")
    print("Estado de las entregas:", entregas["estado"].value_counts().to_dict())
except Exception as error:
    print(f"⚠️ No se pudo cargar la base: {error}")

verificar("lineas_cargadas", "C.0 · líneas cargadas en la base (filas y vacíos)", (len(lineas_detalle), int(lineas_detalle["cantidad_despachada"].isna().sum())), pista="Depende de los pasos B.1, B.4 y B.5.")
verificar("entregas_tarde", "C.0 · entregas tarde", int((entregas["estado"] == "tarde").sum()), pista="Depende del paso B.3 (fechas de entrega).")

def sql(consulta):
    """Ejecuta una consulta sobre cedi.db. Si tiene un error, lo muestra sin detener el cuaderno."""
    try:
        return pd.read_sql(consulta, con)
    except Exception as error:
        print(f"⚠️ La consulta no se pudo ejecutar: {error}")
        return pd.DataFrame()

def revisar_consulta(etiqueta, resultado, columna, clave, consulta="", clave_filtro=None):
    """Revisa que la consulta agrupe por la columna pedida (y use el filtro pedido) sin mostrar la respuesta."""
    try:
        ok_col = columna in resultado.columns and len(resultado) > 0
        ok_val = ok_col and _huella(clave, resultado.iloc[0][columna]) in _HUELLAS[clave]
        ok_fil = True
        if clave_filtro:
            m = re.search(r"ciudad\s*=\s*'([^']*)'", consulta, flags=re.IGNORECASE)
            ok_fil = bool(m) and _huella(clave_filtro, m.group(1)) in _HUELLAS[clave_filtro]
    except Exception:
        ok_col = ok_val = ok_fil = False
    ok = ok_col and ok_val and ok_fil
    _AUTOVERIFICACION[etiqueta] = ok
    if ok:
        print(f"✅ {etiqueta}: la consulta responde lo que se pide.")
    elif not ok_col:
        print(f"⚠️ {etiqueta}: el resultado todavía no tiene la columna «{columna}». Revise el cambio 👉.")
    elif not ok_fil:
        print(f"⚠️ {etiqueta}: el filtro de ciudad todavía no es el que se pide. Revise el cambio 👉.")
    else:
        print(f"⚠️ {etiqueta}: la consulta corre, pero el resultado no coincide. ¿Quedó bien limpia la Parte B?")

Tabla «lineas»:   381 filas · id_linea, id_pedido, sku, cantidad_pedida, cantidad_despachada, descripcion, categoria, proveedor, fecha_pedido, tienda, ciudad
Tabla «entregas»: 130 filas · guia, id_pedido, transportadora, novedad, fecha_compromiso, fecha_entrega, tienda, ciudad, estado, dias_retraso
Estado de las entregas: {'a tiempo': 112, 'tarde': 15, 'sin fecha': 3}
✅ C.0 · líneas cargadas en la base (filas y vacíos): coincide con la cifra de control.
✅ C.0 · entregas tarde: coincide con la cifra de control.


### Cómo se lee una consulta SQL
| Parte | Para qué sirve |
|---|---|
| `SELECT` columnas y cálculos | Qué se muestra. `COUNT(*)` cuenta filas; `SUM(...)` suma; `ROUND(x, 1)` redondea a un decimal |
| `FROM` tabla | De qué tabla salen los datos (`lineas` o `entregas`) |
| `WHERE` condición | Filtra **filas** antes de agrupar |
| `GROUP BY` columna | Forma un grupo por cada valor de esa columna (una fila del resultado por grupo) |
| `ORDER BY` columna | Ordena el resultado; `DESC` = de mayor a menor |

Dos detalles: se escribe `100.0 *` para que la división dé decimales (en SQLite, `20 / 30` da `0`), y `SUM(estado = 'tarde')` cuenta las filas que cumplen la condición. En cada pareja, el 📘 ejemplo ya está escrito; en 👉 su turno se cambia **una palabra**. Cada consulta correcta vale 0,25 y cada lectura, 0,15.

### C.1 · ¿Qué tan completos salen los pedidos?

In [30]:
# C.1 · 📘 Ejemplo resuelto: % despachado por CATEGORÍA
e1 = sql("""
SELECT categoria,
       COUNT(*)                 AS lineas,
       SUM(cantidad_pedida)     AS unidades_pedidas,
       SUM(cantidad_despachada) AS unidades_despachadas,
       ROUND(100.0 * SUM(cantidad_despachada) / SUM(cantidad_pedida), 1) AS pct_despachado
FROM lineas
GROUP BY categoria
ORDER BY pct_despachado          -- de la peor a la mejor
""")
e1

,categoria,lineas,unidades_pedidas,unidades_despachadas,pct_despachado
0,Aseo,102,2671,2343,87.7
1,Abarrotes,126,3292,3241,98.5
2,Bebidas,73,1918,1901,99.1
3,Lácteos,80,2154,2150,99.8


In [31]:
# C.1 · 👉 Su turno: % despachado por PROVEEDOR
# 👉 Cambie la palabra categoria por proveedor en DOS sitios: en el SELECT y en el GROUP BY.
consulta_1 = """
SELECT proveedor,
       COUNT(*)                 AS lineas,
       SUM(cantidad_pedida)     AS unidades_pedidas,
       SUM(cantidad_despachada) AS unidades_despachadas,
       ROUND(100.0 * SUM(cantidad_despachada) / SUM(cantidad_pedida), 1) AS pct_despachado
FROM lineas
GROUP BY proveedor
ORDER BY pct_despachado          -- del peor al mejor
"""
r1 = sql(consulta_1)
revisar_consulta("C.1", r1, "proveedor", "peor_proveedor")
r1

✅ C.1: la consulta responde lo que se pide.


,proveedor,lineas,unidades_pedidas,unidades_despachadas,pct_despachado
0,Aseo Andino del Sur S.A.,53,1356,1039,76.6
1,Alimentos del Opia S.A.S.,71,1870,1843,98.6
2,Distribuciones Magdalena S.A.S.,104,2737,2702,98.7
3,Bebidas Río Claro Ltda.,73,1918,1901,99.1
4,Lácteos Valle Verde S.A.S.,80,2154,2150,99.8


🧠 **Lectura C.1 (0,15).** ¿Qué proveedor despacha peor y cuántos puntos lo separan de los demás? ¿Por qué el ejemplo **por categoría** no mostraba el problema con tanta claridad? (Pista: una categoría puede tener más de un proveedor.)

**Lectura C.1.** Aseo Andino del Sur S.A. es el proveedor con peor cumplimiento: despacha el 76,6% de las 1.356 unidades pedidas, mientras los demás están entre 98,6% y 99,8%; la diferencia frente al siguiente peor es de 22,0 puntos porcentuales (98,6 − 76,6). La vista por categoría mezcla proveedores dentro de cada categoría y puede ocultar que el déficit se concentra en uno de ellos.

### C.2 · ¿Dónde se concentran las entregas tarde?

In [32]:
# C.2 · 📘 Ejemplo resuelto: % de entregas tarde por CIUDAD
e2 = sql("""
SELECT ciudad,
       COUNT(*)                                            AS entregas,
       SUM(estado = 'tarde')                               AS tarde,
       ROUND(100.0 * SUM(estado = 'tarde') / COUNT(*), 1) AS pct_tarde
FROM entregas
WHERE estado <> 'sin fecha'      -- solo los pedidos que ya se entregaron
GROUP BY ciudad
ORDER BY pct_tarde DESC          -- de la peor a la mejor
""")
e2

,ciudad,entregas,tarde,pct_tarde
0,La Plata,24,6,25.0
1,Pitalito,29,6,20.7
2,Garzón,30,2,6.7
3,Neiva,29,1,3.4
4,Campoalegre,15,0,0.0


In [33]:
# C.2 · 👉 Su turno: % de entregas tarde por TRANSPORTADORA
# 👉 Cambie la palabra ciudad por transportadora en DOS sitios: en el SELECT y en el GROUP BY.
consulta_2 = """
SELECT transportadora,
       COUNT(*)                                            AS entregas,
       SUM(estado = 'tarde')                               AS tarde,
       ROUND(100.0 * SUM(estado = 'tarde') / COUNT(*), 1) AS pct_tarde
FROM entregas
WHERE estado <> 'sin fecha'      -- solo los pedidos que ya se entregaron
GROUP BY transportadora
ORDER BY pct_tarde DESC          -- de la peor a la mejor
"""
r2 = sql(consulta_2)
revisar_consulta("C.2", r2, "transportadora", "peor_transportadora")
r2

✅ C.2: la consulta responde lo que se pide.


,transportadora,entregas,tarde,pct_tarde
0,Rápido Sur,38,13,34.2
1,Flota propia,33,1,3.0
2,TransHuila Express,56,1,1.8


🧠 **Lectura C.2 (0,15).** Compare el ejemplo (por ciudad) con su consulta (por transportadora): ¿qué ciudad y qué transportadora salen peor, y con qué porcentajes? Con solo estas dos tablas, ¿se puede saber si la culpa es de la ciudad o de la transportadora? ¿Por qué?

**Lectura C.2.** La Plata es la ciudad con mayor proporción de entregas tarde, con 25,0% (6 de 24), seguida de Pitalito con 20,7% (6 de 29). Rápido Sur es la transportadora con peor resultado general: 34,2% tarde (13 de 38); Flota propia registra 3,0% y TransHuila Express 1,8%. Estas tablas por separado no permiten atribuir la causa, porque cada transportadora puede operar en ciudades diferentes y las condiciones de ruta varían.

### C.3 · La prueba justa: comparar dentro de la misma ciudad
Si dos transportadoras trabajan en la **misma** ciudad, tienen las mismas vías y las mismas distancias. Si una llega tarde mucho más que la otra, la diferencia ya no se puede achacar a la ciudad. `WHERE ciudad = '...'` deja solo las entregas de esa ciudad.

In [34]:
# C.3 · 📘 Ejemplo resuelto: transportadoras dentro de NEIVA
e3 = sql("""
SELECT transportadora,
       COUNT(*)                                            AS entregas,
       SUM(estado = 'tarde')                               AS tarde,
       ROUND(100.0 * SUM(estado = 'tarde') / COUNT(*), 1) AS pct_tarde
FROM entregas
WHERE estado <> 'sin fecha'
  AND ciudad = 'Neiva'           -- solo esta ciudad
GROUP BY transportadora
ORDER BY pct_tarde DESC
""")
e3

,transportadora,entregas,tarde,pct_tarde
0,Flota propia,21,1,4.8
1,TransHuila Express,8,0,0.0


In [35]:
# C.3 · 👉 Su turno: transportadoras dentro de la ciudad que salió PEOR en el ejemplo de C.2
# 👉 Cambie 'Neiva' por esa ciudad, escrita igual que en la tabla (con mayúscula inicial y tilde si la lleva).
consulta_3 = """
SELECT transportadora,
       COUNT(*)                                            AS entregas,
       SUM(estado = 'tarde')                               AS tarde,
       ROUND(100.0 * SUM(estado = 'tarde') / COUNT(*), 1) AS pct_tarde
FROM entregas
WHERE estado <> 'sin fecha'
  AND ciudad = 'La Plata'           -- 👉 cambie la ciudad
GROUP BY transportadora
ORDER BY pct_tarde DESC
"""
r3 = sql(consulta_3)
revisar_consulta("C.3", r3, "transportadora", "peor_transportadora_ciudad", consulta_3, "peor_ciudad")
r3

✅ C.3: la consulta responde lo que se pide.


,transportadora,entregas,tarde,pct_tarde
0,Rápido Sur,15,6,40.0
1,TransHuila Express,9,0,0.0


🧠 **Lectura C.3 (0,15).** En esa ciudad, ¿qué porcentaje de entregas tarde tiene cada transportadora? ¿Qué demuestra esta comparación que no demostraban C.2 por sí solas?

**Lectura C.3.** En La Plata, Rápido Sur tiene 40,0% de entregas tarde (6 de 15), mientras TransHuila Express tiene 0,0% (0 de 9). Esto demuestra que la ciudad no explica por sí sola los retrasos: dentro del mismo destino hay diferencias entre transportadoras. Aun así, hay que tener en cuenta que la muestra es pequeña y no controla todas las diferencias entre rutas y horarios.

## Parte D · Conclusiones para la gerencia (1,3)
La gerente no va a leer este cuaderno: va a leer sus conclusiones. Un buen hallazgo tiene **dato + comparación + qué significa**. Por ejemplo: *«En Neiva, la flota propia llegó tarde en el X % de las entregas, frente al Y % de la otra empresa: en esa ciudad el servicio funciona»*. Una buena recomendación dice **qué** hacer, **quién**, **cuándo** y **con qué indicador** se sabrá si funcionó, y reconoce sus límites.

🧠 **D.1 · ¿Dónde está el problema? (0,4)**
- **(a)** Los pedidos incompletos, ¿son un problema de todo el CEDI o de un proveedor? Respóndalo con una cifra de C.1. *(0,2)*
- **(b)** Las entregas tarde, ¿se deben a la ciudad (distancia, vías) o a la transportadora? Respóndalo con cifras de C.2 y C.3 y explique por qué la comparación **dentro de la misma ciudad** permite separar las dos explicaciones. *(0,2)*

**D.1.** (a) Los pedidos incompletos no parecen ser un problema uniforme del CEDI: Aseo Andino del Sur S.A. solo despacha el 76,6% de las unidades pedidas (1.039 de 1.356), mientras los otros proveedores despachan entre 98,6% y 99,8%. (b) Las entregas tarde se concentran en La Plata (25,0%) y Pitalito (20,7%), y Rápido Sur presenta 34,2% tarde en el total. Para distinguir ciudad y transportadora, la comparación dentro de La Plata revisa a las transportadoras bajo el mismo destino; si sus porcentajes difieren, la ciudad por sí sola no explica el retraso.

🧠 **D.2 · Recomendación para la gerencia (0,5).** En 4 a 6 líneas, recomiende una o dos acciones concretas. Incluya: **qué** se hace y **quién** lo hace (0,15), **cuándo** (0,05), **qué indicador** se vigila y con qué meta, usando las cifras de sus consultas (0,2), y **un límite** del análisis, por ejemplo cuántos meses de datos hay o cuántas líneas se retiraron en la limpieza (0,1).

**D.2.** Durante la próxima semana, la coordinadora de compras y el responsable de Bodega deben revisar con Aseo Andino del Sur los pedidos incompletos y conciliar cantidades pedidas frente a despachadas. Paralelamente, Operaciones debe revisar con Rápido Sur las entregas tardías, empezando por La Plata. Medir semanalmente el porcentaje despachado por proveedor y el porcentaje de entregas tarde por transportadora; meta inicial: elevar al menos al 95% el cumplimiento de Aseo Andino y reducir la tardanza de Rápido Sur por debajo del 20%. El análisis usa solo septiembre de 2026 y retiró 5 líneas imposibles tras eliminar duplicados, por lo que se debe validar con más meses.

🧠 **D.3 · English paragraph (0,4).** Write **one paragraph in English (60 to 100 words)** for the regional manager: the data you used, one cleaning step you applied and why, and your main finding with one number. Use your own words; short and correct sentences are better than long ones.

The analysis used product, order, order-line, and carrier delivery data from September 2026. I removed duplicate order lines and standardized city names so that the same city was not counted under different spellings. The main finding is that Aseo Andino del Sur delivered only 76.6% of the units ordered, while the other suppliers reached at least 98.6%. Rápido Sur also had the highest late-delivery rate, at 34.2%. These results suggest that supplier fulfillment and carrier performance need separate follow-up.

## Cierre · Verificación final y entrega

In [36]:
_claves = [k for k in _AUTOVERIFICACION if k != "prueba"]
print(f"Autoverificaciones en ✅: {sum(_AUTOVERIFICACION[k] for k in _claves)} de {len(_claves)}")
for _k in _claves:
    if not _AUTOVERIFICACION[_k]:
        print("   ⚠️ pendiente:", _k)
print(f"Pasos en la bitácora: {len(bitacora)}")

Autoverificaciones en ✅: 16 de 16
Pasos en la bitácora: 10


### Antes de entregar
- [* ] Escribió su **nombre completo** en la celda de la portada.
- [ *] Ejecutó **todo** el cuaderno en orden (**Entorno de ejecución → Ejecutar todas**) y ninguna celda terminó en error.
- [ *] Respondió **todas** las preguntas 🧠: A.7, B.7, las tres lecturas de la Parte C, D.1, D.2 y el párrafo en inglés D.3.

### Cómo entregar
1. Ponga su nombre al archivo: en Colab, clic en el nombre del cuaderno (arriba a la izquierda), por ejemplo `parcial-c2-ApellidoNombre.ipynb`.
2. Descárguelo **con los resultados**: **Archivo → Descargar → Descargar .ipynb**. En VS Code, guarde con `Ctrl + S`.
3. Opcional: guarde también un PDF del cuaderno ejecutado (**Archivo → Imprimir → Guardar como PDF**).
4. Entre a **Moodle → Ciencia de Datos → tarea «Parcial práctico · Corte 2»** y suba el `.ipynb` (y el PDF, si lo generó).
5. **Plazo: lunes 12 de octubre de 2026, 23:59.** No se reciben entregas tardías: no deje la subida para el último minuto.